# Глубинное обучение на табличных данных

Начнем конечно с импортов различных, которые понадобятся далее. Также, многое отсюда было реализовано с помощью функций копипаста с семинаров, что мы проходили, помимо этого, семинары дали вдохновление на именно такой пайплайн действий) Стоило упомянуть

In [3]:
import pandas as pd
import numpy as np
import random

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader #чтобы подавать данные батчами

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score #для подсчета качества

Зафиксируем сиды

In [ ]:
def seed_everything(seed=42): # чтобы запуски были стабильными и воспроизводимымми
    random.seed(seed) # фиксируем генератор случайных чисел
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    
seed_everything(42)

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device # то есть, если есть видеокарта то юзаем cuda, если нет - то юзаем cpu

device(type='cpu')

Загрузим все предобработанные данные 

In [7]:
X_train =pd.read_csv('../data/X_train.csv')
X_test =pd.read_csv('../data/X_test.csv')
y_train =pd.read_csv('../data/y_train.csv')
y_test =pd.read_csv('../data/y_test.csv')

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((2973, 521), (744, 521), (2973, 1), (744, 1))

Переведем все в тензоры

In [8]:
X_train_tensor =torch.FloatTensor(X_train.values)
X_test_tensor =torch.FloatTensor(X_test.values)
y_train_tensor =torch.FloatTensor(y_train.values)
y_test_tensor =torch.FloatTensor(y_test.values)

X_train_tensor.shape, y_train_tensor.shape

(torch.Size([2973, 521]), torch.Size([2973, 1]))

Теперь соединим это обратно в датасеты (train/test), чтобы передавать в модели именно все признаки с таргетом, после чего создадим dataloader чтобы подавать в модель данные батчами/частями

In [10]:
train_ds = TensorDataset(X_train_tensor,y_train_tensor)
test_ds = TensorDataset(X_test_tensor,y_test_tensor)

train_ds

In [13]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True) #shuffle тут чтобы между эпохами перемешивались строки и модель не привыкала к порядку
test_loader = DataLoader(test_ds, batch_size=256)


In [ ]:
input_size = X_train.shape[1]
input_size 

521

Сейчас построим достаточно базовую модель (вдохновление от 15 семинара). Архитектура простая - 3 полносвязных слоя и все. Далее будем строить несколько более сложных архитектур. Сначала простую чтобы понимать, помогают ли нам новые архитектуры улучшить так скажем 'базовое' качество. 

In [ ]:
class Simple(nn.Module): # наследуемся от класса nn.Module
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 128) # первый скрытый слой - 128 нейронов
        self.fc2 = nn.Linear(128, 64) # второй скрытый - 64
        self.fc3 = nn.Linear(64, 1) # третий  выходной- 1 прогноз
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x) 
        return x

Теперь построим более сложную (чуть) архитектуру. Попробуем добавить просто больше скрытых слоев и добавим еще больше нейронов 

Но, при этом, есильно выше риск переобучения, поэтому будем все это сравнивать на качестве теста

In [18]:
class Deep(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        
        self.fc1 = nn.Linear(input_size, 512) #теперь начинаем уменьшение размерности с 512 
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = self.fc5(x)
        return x

И, добавим третью модель, тут мы добавим еще дропаут и батчнорм. Первый будет бороться с переобучением за счет отключения определенного количества нейронов во время обучения. Батчнорм будет просто стабилизировать значения внутри срктытых слоев

In [19]:
class Regularized(nn.Module):
    def __init__(self, input_size):
        super().__init__()


        self.net = nn.Sequential(nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2), #1 скрытый 
                                 nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),nn.Dropout(0.2),  #2 скрытый
                                 nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))    #3 скрытый и 4 выходной
        #sequential тут потому что во первых мы так работали на семинарах, а во-вторых потому что слои и операции в них просто идут по порядку
        #без сложностей , и следующая функция красивая в 2 строки получается )))

    def forward(self, x):
        return self.net(x)

Далее, переходим к следующей части , пока просто зададим функцию потерь и потом напишем фукнцию обучения

In [20]:
criterion = nn.MSELoss()

In [21]:
from tqdm import tqdm

def train(model, device, train_loader, optimizer, criterion, epoch): 
    model.train()
    train_loss = 0
    
    for data, target in tqdm(train_loader):
        data = data.to(device)
        target = target.to(device)
        
        optimizer.zero_grad() #обнуяем градиенты
        
        output = model(data) #прямой проход
        loss = criterion(output, target) #считаем ошибку
        
        loss.backward() #обратынй проход
        optimizer.step() #шаг оптимизатором
        train_loss += loss.item()
    
    train_loss = train_loss / len(train_loader)
    
    tqdm.write('\nTrain set: Average loss: {:.4f}'.format(train_loss)) #как в семе, но чуть поправленная, потому что там для картинок
    return train_loss

Теперь функция тестирования, оч похожа на сем, но там картинки

In [22]:
def test(model, device, test_loader, criterion):
    model.eval()  
    
    test_loss = 0
    preds = []
    true = []
    
    # показываем, что обучения нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            
            output = model(data)
            test_loss += criterion(output, target).item()
            
            preds.append(output.cpu())
            true.append(target.cpu())
    
    test_loss = test_loss / len(test_loader)
    
    preds = torch.cat(preds).numpy().ravel() #тут мы клеим прогнозы в одномерный массив нампай 
    true = torch.cat(true).numpy().ravel()
    
    preds = np.expm1(preds) #возвращаем логарифмированные числа в первоначальное 
    true = np.expm1(true)
    
    mae = mean_absolute_error(true, preds)
    rmse = np.sqrt(mean_squared_error(true, preds))
    r2 = r2_score(true, preds)
    
    tqdm.write('Test set: Average loss: {:.4f}, MAE: {:.2f}, RMSE: {:.2f}, R2: {:.4f}'.format(
        test_loss, mae, rmse, r2))
    
    return test_loss, mae, rmse, r2

Теперь нужна функция запуска наших экспериментов, и после этого можно будет лицезреть качество выстроенных нами моделей))

In [ ]:
# # основная функция для экспериментов
# def main(model, model_name):
#     seed_everything(42)  # фиксируем сиды
    
#     model = model.to(device)
#     optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
#     train_losses = []
    
#     for epoch in range(1, 16):
#         print('\nEpoch:', epoch)
#         train_loss = train(model, device, train_loader, optimizer, criterion, epoch)
#         train_losses.append(train_loss)
#         test_loss, mae, rmse, r2 = test(model, device, test_loader, criterion)
    
#     print('Training is ended!')
    
#     result = {'model': model_name, 'test_loss': test_loss,'MAE': mae,'RMSE': rmse, 'R2': r2}
    
#     return model, train_losses, result

In [26]:
# основная функция для экспериментов
def main(model):
    seed_everything(42)  # фиксируем сиды
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    train_losses = []
    test_losses = []

    for epoch in range(1, 16):
        print('\nEpoch:', epoch)
        train_loss = train(model, device, train_loader, optimizer, criterion, epoch)
        test_loss, mae, rmse, r2 = test(model, device, test_loader, criterion)
        train_losses.append(train_loss)
        test_losses.append(test_loss)
    
    print('Training is ended!')
    
    
    return model, train_losses, test_losses, test_loss, mae, rmse, r2

In [27]:
main(Simple(input_size))


Epoch: 1


  0%|          | 0/47 [00:00<?, ?it/s]

100%|██████████| 47/47 [00:00<00:00, 1171.17it/s]



Train set: Average loss: 164.0120
Test set: Average loss: 17.6791, MAE: 11100128.00, RMSE: 16146586.30, R2: -1.7310

Epoch: 2


100%|██████████| 47/47 [00:00<00:00, 1455.08it/s]


...

100%|██████████| 47/47 [00:00<00:00, 1402.69it/s]



Train set: Average loss: 0.3175
Test set: Average loss: 0.6330, MAE: 3936181.00, RMSE: 11099798.00, R2: -0.2906

Epoch: 15


100%|██████████| 47/47 [00:00<00:00, 1371.82it/s]


Train set: Average loss: 0.3029
Test set: Average loss: 0.7207, MAE: 4684155.50, RMSE: 18121648.35, R2: -2.4400
Training is ended!


(Simple(
   (fc1): Linear(in_features=521, out_features=128, bias=True)
   (fc2): Linear(in_features=128, out_features=64, bias=True)
   (fc3): Linear(in_features=64, out_features=1, bias=True)
 ),
 [164.01196305295255,
  5.234250267769428,
  1.3508062844580793,
  0.7764220519902858,
  0.5250781207008565,
  0.409910926952007,
  0.31107982327329353,
  0.2504280238075459,
  0.2468750239052671,
  0.25008701548931445,
  0.26374585593634464,
  0.28502300635297245,
  0.3073674925464265,
  0.31753918560261424,
  0.30290067195892334],
 [17.679129282633465,
  2.1792651414871216,
  1.1993541717529297,
  0.9718583424886068,
  0.8419122497240702,
  0.7737265626589457,
  0.728002925713857,
  0.7459394137064616,
  0.6611213286717733,
  0.7598478297392527,
  0.63221076130867,
  0.7151290277640024,
  0.6950763861338297,
  0.6330266694227854,
  0.7207174003124237],
 0.7207174003124237,
 4684155.5,
 np.float64(18121648.351522993),
 -2.4400389194488525)

In [28]:
main(Deep(input_size))


Epoch: 1


100%|██████████| 47/47 [00:00<00:00, 659.27it/s]



Train set: Average loss: 80.0239
Test set: Average loss: 15.0257, MAE: 8003356949741568.00, RMSE: 137389197422626288.00, R2: -197730260164724916224.0000

Epoch: 2


100%|██████████| 47/47 [00:00<00:00, 793.00it/s]



Train set: Average loss: 8.5533
Test set: Average loss: 2.9902, MAE: 1610864000.00, RMSE: 41635958165.72, R2: -18159552.0000

Epoch: 3


...

100%|██████████| 47/47 [00:00<00:00, 785.57it/s]



Train set: Average loss: 0.2356
Test set: Average loss: 1.0656, MAE: 4470363.00, RMSE: 23958904.19, R2: -5.0131

Epoch: 15


100%|██████████| 47/47 [00:00<00:00, 787.73it/s]


Train set: Average loss: 0.3168
Test set: Average loss: 0.9502, MAE: 4249598.00, RMSE: 12986282.57, R2: -0.7666
Training is ended!


(Deep(
   (fc1): Linear(in_features=521, out_features=512, bias=True)
   (fc2): Linear(in_features=512, out_features=256, bias=True)
   (fc3): Linear(in_features=256, out_features=128, bias=True)
   (fc4): Linear(in_features=128, out_features=64, bias=True)
   (fc5): Linear(in_features=64, out_features=1, bias=True)
 ),
 [80.02390324815791,
  8.553295107598,
  1.250685324694248,
  0.43087673789643227,
  0.2709488730798376,
  0.21334784573062937,
  0.1927254266561346,
  0.18010850567766962,
  0.17982653964390147,
  0.18010035339505115,
  0.20125052277395067,
  0.2213095263280767,
  0.20160872298986354,
  0.23561471470809997,
  0.3168444985404928],
 [15.025650342305502,
  2.990152676900228,
  1.3366732597351074,
  1.1881693998972576,
  1.0697641173998516,
  1.0610308448473613,
  1.0488410393397014,
  1.0533809661865234,
  0.9574244419733683,
  1.0152660012245178,
  0.8998761773109436,
  1.0185068249702454,
  0.84211665391922,
  1.0655629833539326,
  0.9502490758895874],
 0.95024907588958

In [29]:
main(Regularized(input_size))


Epoch: 1


100%|██████████| 47/47 [00:00<00:00, 576.54it/s]



Train set: Average loss: 141.5580
Test set: Average loss: 34.4700, MAE: 10898243.00, RMSE: 14543069.35, R2: -1.2155

Epoch: 2


100%|██████████| 47/47 [00:00<00:00, 666.56it/s]



Train set: Average loss: 5.4741
Test set: Average loss: 3.1703, MAE: 9591458.00, RMSE: 36040430.05, R2: -12.6065

Epoch: 3


...

100%|██████████| 47/47 [00:00<00:00, 679.19it/s]



Train set: Average loss: 2.3888
Test set: Average loss: 0.9888, MAE: 7689612.50, RMSE: 35664076.87, R2: -12.3239

Epoch: 15


100%|██████████| 47/47 [00:00<00:00, 675.41it/s]


Train set: Average loss: 2.3609
Test set: Average loss: 2.2027, MAE: 7803639.00, RMSE: 16612241.38, R2: -1.8908
Training is ended!


(Regularized(
   (net): Sequential(
     (0): Linear(in_features=521, out_features=256, bias=True)
     (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (2): ReLU()
     (3): Dropout(p=0.2, inplace=False)
     (4): Linear(in_features=256, out_features=128, bias=True)
     (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (6): ReLU()
     (7): Dropout(p=0.2, inplace=False)
     (8): Linear(in_features=128, out_features=64, bias=True)
     (9): ReLU()
     (10): Linear(in_features=64, out_features=1, bias=True)
   )
 ),
 [141.5579914336509,
  5.474079233534793,
  3.5942109747135893,
  3.186114118454304,
  2.9358914278923196,
  2.815400463469485,
  2.9847504483892564,
  2.6388885442246783,
  2.5927336317427616,
  2.5373176463106843,
  2.6443543687779854,
  2.4534556688146387,
  2.3446633714310665,
  2.3887615254584778,
  2.3608938582400056],
 [34.469966888427734,
  3.1702696482340493,
  2.900392770767212,
 